## Fix route ruta directa

Este notebook limpia las rutsa de ruta directa, pasa a ser lineas sobrelapeadas a una sola linea por ruta sin sobrelape

![rutas](assets/depic_cambio.png)

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
path_gtfs = "../1-scraping_ruta_directa/data/proc/rutas_procesadas_tampico_1.geojson"

In [3]:
routes = gpd.read_file(path_gtfs)
routes.head()

,data.route.shortName,data.route.longName,shape_id,geometry
0,74,Isleta Pérez,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


## Calcular longitud

In [4]:
import geopandas as gpd

# 1️⃣ Asegúrate del CRS actual
print(routes.crs)
# Si dice EPSG:4326 (lat/lon), proyecta a UTM (para México, por ejemplo zona 14N)
routes_m = routes.to_crs(32614)

# 2️⃣ Calcula la longitud (en metros)
routes_m["length_m"] = routes_m.length

# 3️⃣ (Opcional) también en kilómetros
routes_m["length_km"] = routes_m["length_m"] / 1000

# 4️⃣ Muestra resultado
routes_m = routes_m.sort_values("length_m", ascending=False).reset_index(drop=True)
routes_m.head(40)

EPSG:4326


,data.route.shortName,data.route.longName,shape_id,geometry,length_m,length_km
0,108,Cuauhtémoc - Tampico - Universidad Politécnica...,shape_108,"LINESTRING (586676.687 2493248.966, 587405.453...",114364.339311,114.364339
1,108A,Cuauhtémoc - Tampico - Univ. Politécnica de Al...,shape_108A,"LINESTRING (586674.631 2493248.954, 587420.891...",113857.569719,113.857570
2,111A,Tampico - Colonias - El Fuerte - Penal por Av....,shape_111A,"LINESTRING (595215.577 2481185.929, 595276.649...",101312.989576,101.312990
3,114,López Mateos - Margaritas - Tampico por Av. Hi...,shape_114,"LINESTRING (595812.56 2493252.291, 595203.134 ...",99403.236689,99.403237
4,11,Tampico - Colonias - El Fuerte - Penal por Av....,shape_11,"LINESTRING (595215.577 2481185.929, 595276.649...",99266.776666,99.266777
5,114A,López Mateos - Margaritas - Tampico por Ayunta...,shape_114A,"LINESTRING (595812.56 2493252.291, 595203.134 ...",98834.685863,98.834686
6,36,Santa Amalia - Polvorín - Madero -,shape_36,"LINESTRING (617669.347 2456914.141, 617937.312...",88190.919554,88.190920
7,107,Santa Amalia - Tampico por Av. Hidalgo y Ayunt...,shape_107,"LINESTRING (605690.245 2479892.31, 605911.353 ...",66869.299309,66.869299
8,103B,Electricistas - Arboledas - Tampico por H. Ayu...,shape_103B,"LINESTRING (609746.409 2479210.919, 609615.563...",66000.758629,66.000759
9,107A,Santa Amalia - Tampico por Ayuntamiento y Av....,shape_107A,"LINESTRING (605690.245 2479892.31, 605911.353 ...",64805.005254,64.805005


## Guardar en archivos separados

In [6]:
import geopandas as gpd
import os

# Asegúrate de tener tu GeoDataFrame cargado
# routes = gpd.read_file("rutas.shp")  # o como sea que lo obtuviste

# Crea una carpeta de salida
output_folder = "./data/routes/rutas_individuales"
os.makedirs(output_folder, exist_ok=True)

# Itera por cada fila y guarda en un archivo distinto
for idx, row in routes.iterrows():
    shape_id = row["shape_id"]
    ruta = gpd.GeoDataFrame([row], columns=routes.columns, crs=routes.crs)

    # Archivo GeoJSON (puedes cambiar a .shp o .gpkg)
    output_path = os.path.join(output_folder, f"{shape_id}.geojson")
    ruta.to_file(output_path, driver="GeoJSON")

    print(f"✅ Guardado: {output_path}")
    
    import geopandas as gpd
import os

# Carpeta donde estaban los archivos originales
input_folder = "./data/routes/rutas_individuales"
# Carpeta para los archivos vacíos
output_folder = "./data/routes/rutas_vacias"
os.makedirs(output_folder, exist_ok=True)

# Recorre los nombres de los archivos existentes
for filename in os.listdir(input_folder):
    print(input_folder)
    if filename.endswith(".geojson"):
        shape_id = os.path.splitext(filename)[0]  # quita la extensión

        # Crear GeoDataFrame vacío con misma estructura
        empty_gdf = gpd.GeoDataFrame(columns=["data.route.shortName", "data.route.longName", "shape_id", "geometry"],
                                     geometry="geometry", crs="EPSG:4326")

        # Guardar el archivo vacío con el mismo nombre
        output_path = os.path.join(output_folder, f"{shape_id}.geojson")
        empty_gdf.to_file(output_path, driver="GeoJSON")

        print(f"🕳️ Archivo vacío generado: {output_path}")

✅ Guardado: ./data/routes/rutas_individuales/shape_74.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_76.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_81.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_89A.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_67.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_82.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_66.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_75.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_59.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_56.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_73.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_85.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_71.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_69.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_72.geojson
✅ Guardado: ./data/routes/rutas_individuales/shape_88.geojson
✅ Guard

,data.route.shortName,data.route.longName,shape_id,geometry,length_m,length_km
53,119,Pedrera - Soriana - Aeropuerto,shape_119,"LINESTRING (615436.96 2475730.583, 615465.083 ...",33696.444991,33.696445


In [68]:
routes_select = routes_m.iloc[1:2,:]
routes_select.head()


,data.route.shortName,data.route.longName,shape_id,geometry,length_m,length_km
1,108A,Cuauhtémoc - Tampico - Univ. Politécnica de Al...,shape_108A,"LINESTRING (586674.631 2493248.954, 587420.891...",113857.569719,113.85757


In [76]:
routes_select = routes_m[routes_m["data.route.shortName"] == "119"]

In [77]:


# 3) Revisa las razones de fallo
print(res["reason"].value_counts(dropna=False).head(10))
print(res[res["reason"].notna()].head(10)[["reason"]])

# 4) Si ahora hay válidas:
centerlines = centerlines_ok
print(centerlines.head())

NameError: name 'res' is not defined

In [78]:
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import linemerge

# Suponiendo que ya tienes:
# routes = routes.head(1)

# 1) Tomar la geometría y garantizar que sea un LineString
geom = routes_select.geometry.iloc[0]
if geom.geom_type == "MultiLineString":
    m = linemerge(geom)
    # si sigue siendo Multi, usa el tramo más largo
    if m.geom_type == "MultiLineString":
        geom = max(list(m.geoms), key=lambda g: g.length)
    else:
        geom = m

assert geom.geom_type == "LineString", "La geometría debe ser LineString o convertible a LineString."

# 2) Elegir n (número de segmentos)
n = 50  # <-- cámbialo a lo que necesites

# 3) Interpolar puntos y construir segmentos
pts = [geom.interpolate(geom.length * i / n) for i in range(n + 1)]
segs = [LineString([pts[i], pts[i+1]]) for i in range(n)]

# 4) Crear GeoDataFrame con 'order' y copiar atributos originales
row = routes_select.iloc[0].drop(labels=["geometry"])
seg_gdf = gpd.GeoDataFrame(
    [row.to_dict() for _ in range(n)],  # duplica atributos
    geometry=segs,
    crs=routes_select.crs
)
seg_gdf["order"] = range(1, n + 1)

# 5) (Opcional) guardar
seg_gdf.to_file(f"{routes_select.iloc[0]['shape_id']}_segments.geojson", driver="GeoJSON")

seg_gdf.head()

,data.route.shortName,data.route.longName,shape_id,length_m,length_km,geometry,order
0,119,Pedrera - Soriana - Aeropuerto,shape_119,33696.444991,33.696445,"LINESTRING (615436.96 2475730.583, 615029.639 ...",1
1,119,Pedrera - Soriana - Aeropuerto,shape_119,33696.444991,33.696445,"LINESTRING (615029.639 2476023.081, 614800.094...",2
2,119,Pedrera - Soriana - Aeropuerto,shape_119,33696.444991,33.696445,"LINESTRING (614800.094 2476548.826, 614744.118...",3
3,119,Pedrera - Soriana - Aeropuerto,shape_119,33696.444991,33.696445,"LINESTRING (614744.118 2477064.113, 614071.154...",4
4,119,Pedrera - Soriana - Aeropuerto,shape_119,33696.444991,33.696445,"LINESTRING (614071.154 2477028.058, 613745.363...",5


### Hacer buffer

| Región                  | EPSG (UTM) |
|--------------------------|-------------|
| Occidente (Guadalajara) | 32613       |
| Centro (CDMX, Puebla)   | 32614       |
| Oriente (Veracruz)      | 32615       |

In [80]:
seg_gdf = seg_gdf.to_crs(epsg=32615) 
#seg_gdf["geometry"] = seg_gdf["geometry"].buffer(5)
seg_gdf.to_file(f"./temp_segments_buffered.geojson", driver="GeoJSON") 
seg_gdf.explore(column="order")

### obtener linea a partir de buffer